# 23-24. Building a synthetic version of a real customer table

This is the hands-on companion to Week 5's synthetic data lessons. Those lessons covered when synthetic data makes sense, how a model actually learns a table, and how to judge whether the result is good enough. This notebook builds one, on a real dataset, start to finish: load real data, preprocess it, train and tune CTGAN and TVAE, generate a synthetic version, and check whether it actually works for the thing we built it for.

**Time**: 45-90 minutes, most of it spent waiting on training, not writing code
**Cost**: $0. Everything here runs on your own computer. No API key, no billing, nothing metered.

> **Disclaimer.** This is educational guidance, not a production recipe. `auto_synthetic_data_platform` is not an official Google product and has had a single release since February 2024. This notebook includes the exact install steps needed to get it running on a current Python, verified end to end, but expect to revisit them if the underlying packages move again. A real production pipeline needs a real review by someone with the relevant expertise.

## Setting up your environment

Run this before anything else in the notebook. This week's package, [Auto Synthetic Data Platform](https://github.com/google-marketing-solutions/auto_synthetic_data_platform), wraps a library called [synthcity](https://github.com/vanderschaarlab/synthcity) to provide the actual CTGAN and TVAE models this notebook trains. The platform's last release, from February 2024, locks in exact versions of `numpy`, `torch`, and a few other libraries that no longer exist for a current Python. The project file tells `uv` to install the package and ignore those broken pins. Tested on Python 3.11 and 3.12. It does not work on Python 3.13 or newer: synthcity needs a version of torch that was never built for 3.13. The setup cell below checks for this and stops with instructions instead of failing halfway through.

This course uses one environment: the `.venv` folder next to `pyproject.toml`. From that project folder, run:

```bash
uv sync
```

Then point this notebook's kernel at that environment:
- **VS Code / Cursor**: **Select Kernel** (top right) → **Python Environments** → the `.venv` in the project folder (`.venv\Scripts\python.exe` on Windows).
- **PyCharm**: **Settings → Project → Python Interpreter → Add Interpreter → Existing** → `.venv/bin/python` (`.venv\Scripts\python.exe` on Windows).

Full walkthrough of `uv`, including a browser-tab alternative that skips the IDE step entirely, is in [Week 2's setup guide](../week_2/lesson08_setup_guide.ipynb). The code cell below checks that the package is installed. If it is missing, run `uv sync` from the project folder, select that kernel, and run the cell again.

In [1]:
# Must be the very first thing this kernel runs, before torch/xgboost/numpy's BLAS
# backend initialize their own thread pools for the first time in this process.
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

# synthcity can't be installed on Python 3.13+: its torch pin has no 3.13 wheels.
import sys
if sys.version_info >= (3, 13):
    _version = f"{sys.version_info.major}.{sys.version_info.minor}"
    raise RuntimeError(
        f"This notebook needs Python 3.11 or 3.12, but this kernel is Python {_version}. "
        "From the project folder run `uv sync`, then select the .venv kernel "
        "(.venv\\Scripts\\python.exe on Windows)."
    )

import importlib.util
if importlib.util.find_spec("auto_synthetic_data_platform") is None:
    raise RuntimeError(
        "auto_synthetic_data_platform is not installed in this kernel. "
        "From the project folder (the one with pyproject.toml) run `uv sync`, "
        "select the .venv kernel, and run this cell again."
    )
print("auto_synthetic_data_platform is installed.")

auto_synthetic_data_platform is installed.


In [2]:
# This cell doesn't install anything -- it just tells you whether the cell
# above actually worked, in plain language, instead of you having to read
# through pip's output to find out.
try:
    from auto_synthetic_data_platform import preprocessing
    from auto_synthetic_data_platform import numerical_variables_processing
    from auto_synthetic_data_platform import synthetic_data_model_tuning as sdmt
    from synthcity.plugins import Plugins
    from synthcity.plugins.core import dataloader
    import pandas as pd
    import numpy as np
    import pathlib
    import torch
    from synthcity.utils.constants import DEVICE
    if not torch.cuda.is_available():
        raise RuntimeError(
            "PyTorch cannot see the NVIDIA GPU. From the project folder run `uv sync`, "
            "restart this kernel, and select the project .venv."
        )
    print(f"Training will use {torch.cuda.get_device_name(0)} ({DEVICE}).")
    print("Setup complete. Everything needed for this notebook imported correctly.")
except (ImportError, ValueError) as e:
    print("Something didn't install correctly:", e)
    print("Try running the cell above again first. If the error mentions numpy.dtype size,")
    print("restart the kernel and run this cell again.")
    print("If it still fails: restart the kernel, then run both cells again from the top.")
    print("Make sure you're running this notebook inside the virtual environment described above.")

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2+cu121
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


[KeOps] Warning : CUDA libraries not found or could not be loaded; Switching to CPU only.
Training will use NVIDIA GeForce RTX 3070 Laptop GPU (cuda).
Setup complete. Everything needed for this notebook imported correctly.


> **About version warnings.** The published package asks for exact old versions such as `numpy==1.23.5`. Those builds no longer install, so `uv sync` ignores them and installs a synthcity release that does install. As long as the cell after the setup cell prints "Setup complete," you're fine.

---

## The dataset

The analysis this session builds toward is the one named in the lessons: a propensity model predicting whether a customer buys again within 90 days. That needs a real dataset with real repeat-purchase behaviour in it, not an invented one.

This notebook uses the [Olist Brazilian e-commerce dataset](https://huggingface.co/datasets/miminmoons/olist-ecommerce-for-delivery-and-review-prediction): about 100,000 real, anonymised orders placed at a Brazilian online marketplace between September 2016 and September 2018. It's rich enough to build a genuine customer-level table from, and it's real, so every number in this notebook, including the uncomfortable ones, reflects actual purchase behaviour rather than something invented to make the example work.

Two of the columns the lessons used as illustration don't exist in Olist at all, and it's worth naming the substitution rather than quietly working around it:

- Olist has no age data of any kind, so **age band** is dropped. In its place, this notebook uses **top product category**, the category a customer bought from most, since it's a real column with genuine marketing value (it's exactly the kind of thing a personalisation model would use).
- Olist has no marketing-channel or acquisition-source data, so **channel of first visit** is replaced with **primary payment type** (credit card, boleto, voucher, debit card), the closest real categorical column available.

Everything else is exactly what the lessons describe: region (the customer's state), number of orders, average basket value, days since the last order, and the target, whether the customer bought again within 90 days.

### Avoiding a leak before it happens

The naive way to build this table, count each customer's total orders and check whether their *last* order was followed by another one, leaks the future into the present: a customer's own final order count already tells you whether they bought again. This notebook instead fixes a single cutoff date late enough that every customer has a full, uncensored 90-day window after it, and splits strictly on that date: every feature (region, order count, basket value, days since last order, payment type, category) is computed only from orders *before* the cutoff, and the label looks only at orders *after* it. That's the only way "bought again within 90 days" means what it says.

In [3]:
import pandas as pd
import numpy as np
from huggingface_hub import hf_hub_download
import shutil, os

REPO = "miminmoons/olist-ecommerce-for-delivery-and-review-prediction"
FILES = [
    "data/olist_customers_dataset.csv",
    "data/olist_orders_dataset.csv",
    "data/olist_order_items_dataset.csv",
    "data/olist_order_payments_dataset.csv",
    "data/olist_products_dataset.csv",
]

os.makedirs("data", exist_ok=True)
for f in FILES:
    path = hf_hub_download(repo_id=REPO, filename=f, repo_type="dataset")
    shutil.copy(path, os.path.join("data", os.path.basename(f)))

customers = pd.read_csv("data/olist_customers_dataset.csv")
orders = pd.read_csv("data/olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
items = pd.read_csv("data/olist_order_items_dataset.csv")
payments = pd.read_csv("data/olist_order_payments_dataset.csv")
products = pd.read_csv("data/olist_products_dataset.csv")
print("loaded:", len(customers), "customers,", len(orders), "orders")

olist_customers_dataset.csv:   0%|          | 0.00/9.03M [00:00<?, ?B/s]

data/olist_orders_dataset.csv: reconstructing file:   0%|          |  0.00B / 17.7MB            

data/olist_orders_dataset.csv: downloading bytes:           |  0.00B            

data/olist_order_items_dataset.csv: reconstructing file:   0%|          |  0.00B / 15.4MB            

data/olist_order_items_dataset.csv: downloading bytes:           |  0.00B            

olist_order_payments_dataset.csv:   0%|          | 0.00/5.78M [00:00<?, ?B/s]

olist_products_dataset.csv:   0%|          | 0.00/2.38M [00:00<?, ?B/s]

loaded: 99441 customers, 99441 orders


In [4]:
# Cancelled orders never delivered anything, so they carry no purchase signal.
orders = orders[orders["order_status"] != "canceled"]

# One payment value per order (payments can be split across installments/methods),
# plus whichever payment type carried the largest share of that order.
order_value = payments.groupby("order_id")["payment_value"].sum().rename("order_value")
order_payment_type = (
    payments.sort_values("payment_value", ascending=False)
    .groupby("order_id")["payment_type"].first()
    .rename("payment_type")
)

# The product category with the most items in each order.
items_with_cat = items.merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
order_top_category = (
    items_with_cat.groupby("order_id")["product_category_name"]
    .agg(lambda s: s.value_counts().idxmax() if s.notna().any() else np.nan)
    .rename("top_category")
)

o = orders.merge(customers[["customer_id", "customer_unique_id", "customer_state"]], on="customer_id")
o = o.merge(order_value, on="order_id", how="left")
o = o.merge(order_payment_type, on="order_id", how="left")
o = o.merge(order_top_category, on="order_id", how="left")
o = o.sort_values(["customer_unique_id", "order_purchase_timestamp"])

# Cutoff: 90 days before the last order in the dataset, so the forward window
# is never censored.
MAX_DATE = o["order_purchase_timestamp"].max()
CUTOFF = MAX_DATE - pd.Timedelta(days=90)
print("last order in the data:", MAX_DATE.date(), " |  cutoff:", CUTOFF.date())

history = o[o["order_purchase_timestamp"] < CUTOFF]
future = o[(o["order_purchase_timestamp"] >= CUTOFF) & (o["order_purchase_timestamp"] < CUTOFF + pd.Timedelta(days=90))]
repeat_customers = set(future["customer_unique_id"].unique())

def most_common(s):
    s = s.dropna()
    return s.value_counts().idxmax() if len(s) else np.nan

customer_table = history.groupby("customer_unique_id").agg(
    region=("customer_state", "first"),
    num_orders=("order_id", "nunique"),
    avg_basket_value=("order_value", "mean"),
    last_order_date=("order_purchase_timestamp", "max"),
    primary_payment_type=("payment_type", most_common),
    top_category=("top_category", most_common),
)
customer_table["days_since_last_order"] = (CUTOFF - customer_table["last_order_date"]).dt.days
customer_table["bought_again_90d"] = customer_table.index.isin(repeat_customers).astype(int)
customer_table = customer_table.drop(columns=["last_order_date"]).reset_index(drop=True)

print("\ncustomer table:", customer_table.shape)
customer_table.head()

last order in the data: 2018-09-03  |  cutoff: 2018-06-05

customer table: (78113, 7)


,region,num_orders,avg_basket_value,primary_payment_type,top_category,days_since_last_order,bought_again_90d
0,SP,1,141.90,credit_card,cama_mesa_banho,25,0
1,SP,1,27.19,credit_card,beleza_saude,28,0
2,SC,1,86.22,credit_card,papelaria,451,0
3,PA,1,43.62,credit_card,telefonia,235,0
4,SP,1,196.89,credit_card,telefonia,202,0


### The real imbalance, before touching a model

Before doing anything else with this table, apply one of the checks this week's lessons will run formally in a moment: how rare is the thing we're trying to predict, really?

In [5]:
positive_rate = customer_table["bought_again_90d"].mean()
print(f"customers: {len(customer_table)}")
print(f"bought again within 90 days: {customer_table['bought_again_90d'].sum()} ({positive_rate:.3%})")

customers: 78113
bought again within 90 days: 418 (0.535%)


That number, well under 1%, is what the platform's own class-imbalance check would call **extreme**. This isn't a flaw in how the table was built. It's what a real 90-day repeat-purchase rate on a real marketplace actually looks like, and it's exactly the scenario the imbalance thresholds in this week's lessons, and the rare-class evaluation lessons from Week 3, were written for.

One tempting shortcut is to keep every customer who bought again and throw away most of the customers who didn't, until the repeat rate looks more like 6%. Training gets easier, but the table is no longer this store. A synthetic copy of that smaller table would describe a marketplace where people come back about ten times as often as they really do. Every later number, including the "will they buy again" model, would be about that invented store.

This notebook keeps every customer. The rare buyers are handled in the job that actually needs to find them, not by deleting rows:

- The generator's job is to copy the real table, including how rare a repeat purchase is. `bought_again_90d` is declared as a category, so CTGAN has to be able to produce a repeat-buyer row, not treat those 418 people as noise. The synthetic table should still come back at about this same tiny rate.
- The prediction model's job, later in the notebook, is to notice those rare buyers. It is told that missing one of them matters in proportion to how rare they are. We judge that model with precision, recall, and F1. Accuracy hides the problem: a model that says "no" for every customer is already right more than 99% of the time.

In [6]:
# Keep every customer. The repeat rate stays the real one, about 0.5%.
working_table = customer_table.reset_index(drop=True)
print("working table:", working_table.shape, " positive rate:", working_table["bought_again_90d"].mean())

working table: (78113, 7)  positive rate: 0.005351221947691166


## Declaring column types

The platform's `Preprocessor` needs to be told, explicitly, which columns are numerical and which are categorical. It will not infer this, and it shouldn't have to: `num_orders`, `avg_basket_value`, and `days_since_last_order` are genuinely numerical, quantities where the distance between two values means something. `region`, `primary_payment_type`, `top_category`, and the target itself are categorical labels, even though `bought_again_90d` happens to be stored as 0 and 1.

In [7]:
column_metadata = {
    "categorical": ["region", "primary_payment_type", "top_category", "bought_again_90d"],
    "numerical": ["num_orders", "avg_basket_value", "days_since_last_order"],
}
EXPERIMENT_DIR = pathlib.Path("experiment")
EXPERIMENT_DIR.mkdir(exist_ok=True)

## Try it: what happens with the platform's defaults

Run the cell below as-is, with the platform's default settings. It will fail. Read the error before moving to the next cell -- figuring out *why* it fails is the actual point of this section.

In [8]:
try:
    pre = preprocessing.Preprocessor(
        dataframe=working_table.copy(),
        experiment_directory=EXPERIMENT_DIR,
        column_metadata=column_metadata,
        # remove_numerical_outliers=True is the platform's own default -- not set explicitly here on purpose.
    )
    pre.output_dataframe
except Exception as e:
    print(f"{type(e).__name__}: {e}")

10/19/2026 20:02:40:WARNING:The input dataframe has missing values in columns: avg_basket_value, primary_payment_type, top_category.
10/19/2026 20:02:40:WARNING:Preprocesssing the missing values using the specified method: 'drop'.


IndexError: index -1 is out of bounds for axis 0 with size 0


### Why: a percentile-based filter breaks on a column that's mostly one number

The platform's default outlier removal trims every numerical column to its 0.5th-99.5th percentile range, then drops anything outside it, using a filter that requires a value to be *strictly greater than* the lower cut and *strictly less than* the upper one. That's a sound approach for `avg_basket_value`, where the extreme values are genuine outliers. It breaks completely on `num_orders`, because in this table, over 90% of customers placed exactly one order before the cutoff -- so both the 0.5th and the 99.5th percentile of that column land on the same number, `1`. A strict filter of `(value > 1) and (value < 1)` is impossible for any value at all, including 1 itself, so the entire table gets dropped, and every downstream step fails on an empty dataframe.

This isn't a hypothetical reading of the source. It's what actually happened when this notebook was built, and it's the reason the five questions from the pipeline-risk-points lesson apply here too: reading what a check actually does, rather than trusting its name, is what surfaces this before it becomes a silent, empty dataframe with no error at all in a slightly less skewed dataset.

The fix: turn the platform's automatic outlier removal off, and apply it by hand only to the one column where it's actually appropriate.

In [9]:
pre = preprocessing.Preprocessor(
    dataframe=working_table.copy(),
    experiment_directory=EXPERIMENT_DIR,
    column_metadata=column_metadata,
    preprocess_metadata=dict(
        remove_duplicates=True,
        preprocess_missing_values=True,
        missing_values_preprocessing_method="drop",
        remove_numerical_outliers=False,
    ),
)
preprocessed = pre.output_dataframe

before = len(preprocessed)
preprocessed = numerical_variables_processing.remove_column_outliers(
    dataframe=preprocessed, column_name="avg_basket_value"
)
print(f"rows after preprocessing + manual outlier trim: {len(preprocessed)} (dropped {before - len(preprocessed)} for basket-value outliers)")
print("positive rate after preprocessing:", preprocessed["bought_again_90d"].mean())

10/19/2026 22:51:25:WARNING:The input dataframe has missing values in columns: avg_basket_value, primary_payment_type, top_category.
10/19/2026 22:51:25:WARNING:Preprocesssing the missing values using the specified method: 'drop'.
10/19/2026 22:51:25:WARNING:The categorical column 'region' dimensionality is 27. The recommended column cardinality range is 50-100. Real and synthetic distribution metrics might get adversely affected.
10/19/2026 22:51:25:WARNING:A moderate class imbalance detected in the 'region' column for variables: RJ, MG, RS, PR, SC, BA, ES, DF, GO, PE, CE, PA.
10/19/2026 22:51:25:WARNING:The categorical column 'primary_payment_type' dimensionality is 4. The recommended column cardinality range is 50-100. Real and synthetic distribution metrics might get adversely affected.
10/19/2026 22:51:25:WARNING:A mild class imbalance detected in the 'primary_payment_type' column for variables: boleto.
10/19/2026 22:51:25:WARNING:A moderate class imbalance detected in the 'primar

rows after preprocessing + manual outlier trim: 74635 (dropped 757 for basket-value outliers)
positive rate after preprocessing: 0.005426408521471159


## The logs, and why they exist

Every check above wrote to a log file, in a workspace folder, not just to this notebook's output. That's deliberate: the whole point of a shareable log is that someone who cannot see the raw table can still see what the preprocessing step found, and help debug a model trained on it, without ever touching the data itself.

Read the actual log below. It should show a missing-value warning for `top_category` (some orders have no matched product), a cardinality note on `region` and `primary_payment_type` (both well under the platform's recommended 50-100 distinct values), and a class-imbalance warning on `bought_again_90d` at the "extreme" level, because the full table still has the real repeat rate of well under 1%.

In [10]:
log_path = sorted(EXPERIMENT_DIR.rglob("*.log"))[0]
print(log_path.read_text())

2026-10-19 20:02:40,694:WARNING : preprocessing : The input dataframe has missing values in columns: avg_basket_value, primary_payment_type, top_category.
2026-10-19 20:02:40,707:WARNING : preprocessing : Preprocesssing the missing values using the specified method: 'drop'.
2026-10-19 22:51:25,479:WARNING : preprocessing : The input dataframe has missing values in columns: avg_basket_value, primary_payment_type, top_category.
2026-10-19 22:51:25,497:WARNING : preprocessing : Preprocesssing the missing values using the specified method: 'drop'.
2026-10-19 22:51:25,508:WARNING : preprocessing : The categorical column 'region' dimensionality is 27. The recommended column cardinality range is 50-100. Real and synthetic distribution metrics might get adversely affected.
2026-10-19 22:51:25,613:WARNING : preprocessing : A moderate class imbalance detected in the 'region' column for variables: RJ, MG, RS, PR, SC, BA, ES, DF, GO, PE, CE, PA.
2026-10-19 22:51:25,616:WARNING : preprocessing : Th

> **Read a log before sending it.** The warnings above name the actual columns and category values that triggered them, which means a "safe to share" log can still carry your column names and some of your category labels. That's a real, if minor, disclosure -- decide whether it matters for a given dataset before it goes external.

---

## Training and auto-tuning a synthetic data model

`Preprocessor` cleaned the table. This section trains a generative model on it and searches for good hyperparameters automatically, rather than hand-tuning them.

A `synthcity` data loader needs to know which column is the prediction target, since several of its evaluation metrics (including the suitability check later in this notebook) train a model on the data and need to know what to predict.

In [11]:
data_loader = dataloader.GenericDataLoader(
    preprocessed, target_column="bought_again_90d", train_size=0.75
)

# One metric per family named in the lessons: a sanity check, a statistical-fidelity
# check, a suitability/performance check, and a privacy check. All four are defined
# so that higher is better, which is what lets the search maximise their mean.
EVALUATION_METRICS = {
    "sanity": ["close_values_probability"],
    "stats": ["inv_kl_divergence"],
    "performance": ["xgb"],
    "privacy": ["k-anonymization"],
}
# The platform's own docs warn this whole search is slow: each trial is a full training
# run, and Optuna is free to sample a large n_iter / wide network on any given trial --
# even on the GPU, one unlucky trial can take a long time. 2 keeps a first run tractable;
# bump this back up (5+) for a real search once you have seen one trial finish.
NUMBER_OF_TRIALS = 2

### Try it: tune TVAE

This runs `NUMBER_OF_TRIALS` full training runs of a tabular variational autoencoder, each with a different sampled set of hyperparameters, and keeps the one that scores best on the four metrics above. Training uses the GPU named in the setup cell. A trial can still take several minutes, and expect real variance between trials: Optuna is free to sample a much larger network or iteration count on any given trial, and one unlucky trial can run far longer than the others. A trial's own progress bar (`n/N [elapsed<remaining, s/it]`) tells you whether it's genuinely working through a large trial or actually stuck; if you want a second, process-level check, `ps -p <kernel PID> -o %cpu,time` a few seconds apart, and confirm `TIME` is still climbing.

In [13]:
tvae_workspace = pathlib.Path("experiment_tvae")
tvae_workspace.mkdir(exist_ok=True)

tvae_model = Plugins().get("tvae", workspace=tvae_workspace)
tvae_tuner = sdmt.SyntheticDataModelTuner(
    data_loader=data_loader,
    synthetic_data_model=tvae_model,
    task_type="classification",
    number_of_trials=NUMBER_OF_TRIALS,
    optimization_direction="maximize",
    evaluation_metrics=EVALUATION_METRICS,
    experiment_directory=tvae_workspace,
)
print("best TVAE hyperparameters:", tvae_tuner.best_hyperparameters)
tvae_tuner.best_synthetic_data_model_evaluation_report

[2026-10-19T23:45:38.556709+0800][21352][CRITICAL] module disabled: d:\Personal project\ai4tm\.venv\Lib\site-packages\synthcity\plugins\generic\plugin_goggle.py
10/19/2026 23:45:38:INFO:Specified 'evaluation_metrics': {'sanity': ['close_values_probability'], 'stats': ['inv_kl_divergence'], 'performance': ['xgb'], 'privacy': ['k-anonymization']}. Optimization direction: 'maximize'.
10/19/2026 23:45:38:INFO:Trial: trial_0. Model: tvae. Selected hyperparameters: {'n_iter': 400, 'lr': 0.001, 'decoder_n_layers_hidden': 3, 'weight_decay': 0.0001, 'batch_size': 512, 'n_units_embedding': 350, 'decoder_n_units_hidden': 100, 'decoder_nonlin': 'leaky_relu', 'decoder_dropout': 0.01916661421562187, 'encoder_n_layers_hidden': 1, 'encoder_n_units_hidden': 300, 'encoder_nonlin': 'tanh', 'encoder_dropout': 0.013817957179661345}
[2026-10-19T23:45:38.687348+0800][21352][CRITICAL] module disabled: d:\Personal project\ai4tm\.venv\Lib\site-packages\synthcity\plugins\generic\plugin_goggle.py
100%|██████████|

best TVAE hyperparameters: {'n_iter': 400, 'lr': 0.001, 'decoder_n_layers_hidden': 3, 'weight_decay': 0.0001, 'batch_size': 512, 'n_units_embedding': 350, 'decoder_n_units_hidden': 100, 'decoder_nonlin': 'leaky_relu', 'decoder_dropout': 0.01916661421562187, 'encoder_n_layers_hidden': 1, 'encoder_n_units_hidden': 300, 'encoder_nonlin': 'tanh', 'encoder_dropout': 0.013817957179661345}


100%|██████████| 400/400 [08:06<00:00,  1.22s/it]


,min,max,mean,stddev,median,iqr,rounds,errors,durations,direction
sanity.close_values_probability.score,0.996409,0.996409,0.996409,0.0,0.996409,0.0,1,0,0.08,maximize
stats.inv_kl_divergence.marginal,0.965955,0.965955,0.965955,0.0,0.965955,0.0,1,0,0.02,maximize
performance.xgb.gt,0.543698,0.543698,0.543698,0.0,0.543698,0.0,1,0,1.22,maximize
performance.xgb.syn_id,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,1.22,maximize
performance.xgb.syn_ood,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,1.22,maximize
performance.xgb_augmentation.gt,0.543698,0.543698,0.543698,0.0,0.543698,0.0,1,0,1.17,maximize
performance.xgb_augmentation.aug_ood,0.000000,0.000000,0.000000,0.0,0.000000,0.0,1,0,1.17,maximize
privacy.k-anonymization.gt,144.000000,144.000000,144.000000,0.0,144.000000,0.0,1,0,0.24,maximize
privacy.k-anonymization.syn,184.000000,184.000000,184.000000,0.0,184.000000,0.0,1,0,0.24,maximize


### Try it: tune CTGAN

Same search, same four metrics, a different model. CTGAN trains two networks against each other (a generator and a discriminator) rather than one autoencoder, and its conditional generation is specifically meant to handle rare categorical values -- worth watching for in the `top_category` and `region` columns, both of which have a long tail of infrequent values.

In [ ]:
ctgan_workspace = pathlib.Path("experiment_ctgan")
ctgan_workspace.mkdir(exist_ok=True)

ctgan_model = Plugins().get("ctgan", workspace=ctgan_workspace)
ctgan_tuner = sdmt.SyntheticDataModelTuner(
    data_loader=data_loader,
    synthetic_data_model=ctgan_model,
    task_type="classification",
    number_of_trials=NUMBER_OF_TRIALS,
    optimization_direction="maximize",
    evaluation_metrics=EVALUATION_METRICS,
    experiment_directory=ctgan_workspace,
)
print("best CTGAN hyperparameters:", ctgan_tuner.best_hyperparameters)
ctgan_tuner.best_synthetic_data_model_evaluation_report

[2026-10-20T01:11:44.121378+0800][21352][CRITICAL] module disabled: d:\Personal project\ai4tm\.venv\Lib\site-packages\synthcity\plugins\generic\plugin_goggle.py
10/20/2026 01:11:44:INFO:Specified 'evaluation_metrics': {'sanity': ['close_values_probability'], 'stats': ['inv_kl_divergence'], 'performance': ['xgb'], 'privacy': ['k-anonymization']}. Optimization direction: 'maximize'.
10/20/2026 01:11:44:INFO:Trial: trial_0. Model: ctgan. Selected hyperparameters: {'generator_n_layers_hidden': 2, 'generator_n_units_hidden': 100, 'generator_nonlin': 'relu', 'n_iter': 500, 'generator_dropout': 0.19746823585430948, 'discriminator_n_layers_hidden': 1, 'discriminator_n_units_hidden': 150, 'discriminator_nonlin': 'relu', 'discriminator_n_iter': 1, 'discriminator_dropout': 0.017505252500753143, 'lr': 0.0001, 'weight_decay': 0.0001, 'batch_size': 500, 'encoder_max_clusters': 6}
[2026-10-20T01:11:45.291994+0800][21352][CRITICAL] module disabled: d:\Personal project\ai4tm\.venv\Lib\site-packages\syn

best CTGAN hyperparameters: {'generator_n_layers_hidden': 2, 'generator_n_units_hidden': 50, 'generator_nonlin': 'relu', 'n_iter': 200, 'generator_dropout': 0.1268754926596946, 'discriminator_n_layers_hidden': 4, 'discriminator_n_units_hidden': 50, 'discriminator_nonlin': 'elu', 'discriminator_n_iter': 4, 'discriminator_dropout': 0.19091773539328805, 'lr': 0.0001, 'weight_decay': 0.0001, 'batch_size': 100, 'encoder_max_clusters': 9}


100%|██████████| 200/200 [1:15:44<00:00, 22.72s/it]


### What the search found

Two plots make the search itself inspectable, rather than a black box that just hands back a winner:

In [ ]:
tvae_tuner.display_parallel_hyperparameter_coordinates()

In [ ]:
tvae_tuner.display_hyperparameter_importances()

The parallel-coordinates plot draws one line per trial across every hyperparameter axis and the resulting score, so a band of lines converging in the same region of an axis shows where the good scores came from. The importances plot ranks each hyperparameter by how much varying it moved the score -- which knobs mattered for this dataset, and which could have been left at their defaults.

### Picking a winner and generating the synthetic table

Compare the two evaluation reports above on the metric that matters most for this week's use case, `performance.xgb` (suitability), and keep whichever model actually did better on the thing this dataset is for. Generate a full-size synthetic table from that model.

In [ ]:
# Compare on the suitability metric specifically -- the one this week's use case
# actually depends on -- rather than the search's blended objective.
tvae_suitability = tvae_tuner.best_synthetic_data_model_evaluation_report.loc["performance.xgb.mean"].iloc[0] \
    if "performance.xgb.mean" in tvae_tuner.best_synthetic_data_model_evaluation_report.index else None
ctgan_suitability = ctgan_tuner.best_synthetic_data_model_evaluation_report.loc["performance.xgb.mean"].iloc[0] \
    if "performance.xgb.mean" in ctgan_tuner.best_synthetic_data_model_evaluation_report.index else None
print("TVAE suitability:", tvae_suitability)
print("CTGAN suitability:", ctgan_suitability)

best_tuner = tvae_tuner if (tvae_suitability or 0) >= (ctgan_suitability or 0) else ctgan_tuner
best_name = "tvae" if best_tuner is tvae_tuner else "ctgan"
print("\nwinner:", best_name)

synthetic_table = sdmt.generate_synthetic_data_with_synthetic_data_model(
    count=len(preprocessed), model=best_tuner.best_synthetic_data_model
)
synthetic_table.to_csv("synthetic_customer_table.csv", index=False)
synthetic_table.head()

---

## Reading the evaluation report

The winning model's evaluation report already showed the four metrics the search optimised against. This section reads a fuller report, and then goes one step further: it actually trains the propensity model this whole exercise was for, once on the synthetic table and once on the real one, and compares them on real, held-out customers.

In [ ]:
full_report = best_tuner.best_synthetic_data_model_full_evaluation_report
full_report

Four families, four different questions:

- **Sanity** (`close_values_probability` and neighbours) asks whether the output resembles real data at all -- the check that catches a model that produced nonsense.
- **Statistical fidelity** (`inv_kl_divergence` and neighbours) asks how closely the synthetic table's distributions match the real one's, column by column and across relationships between columns.
- **Performance / suitability** (`xgb` and neighbours) asks the question this week's use case actually depends on: does a model trained on the synthetic table make accurate predictions on real customers?
- **Privacy** (`k-anonymization` and neighbours) asks how exposed an individual synthetic record is. Higher k means each synthetic customer looks identical, on the attributes checked, to at least k-1 others.

These fight each other by construction. A model that reproduced the real table exactly would score perfectly on fidelity and suitability and offer no privacy at all, because the output would be the input. A model that generated random values would score perfectly on privacy and be useless for anything else. Every usable model sits somewhere between those two, and the "right" point on that trade-off is a judgment about how this specific table will be used, not a number the platform can hand back on its own.

### Suitability, tested directly: does the propensity model actually work?

The evaluation report's `performance.xgb` metric already trains a gradient-boosted model on the synthetic data and tests it on real data. This section does the same thing explicitly, with the metrics Week 3 taught for exactly this kind of rare-outcome problem: precision, recall, and F1, not accuracy, since well under 10% of customers in this table bought again. The table is not rebalanced. Instead the model is given `scale_pos_weight`: missing one repeat buyer counts as heavily as that buyer is rare in the training rows.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import xgboost as xgb

CAT_COLS = ["region", "primary_payment_type", "top_category"]
NUM_COLS = ["num_orders", "avg_basket_value", "days_since_last_order"]
TARGET = "bought_again_90d"

real_train, real_test = train_test_split(
    preprocessed, test_size=0.25, stratify=preprocessed[TARGET], random_state=42
)

def make_pipeline(y):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), CAT_COLS),
        ("num", "passthrough", NUM_COLS),
    ])
    # Repeat buyers are about 1 in 200 rows. scale_pos_weight tells the model
    # that missing one of them counts as much as that rarity: count of
    # non-buyers divided by count of buyers. The table itself stays unchanged.
    y = np.asarray(y)
    n_pos = int((y == 1).sum())
    n_neg = int((y == 0).sum())
    weight = (n_neg / n_pos) if n_pos else 1.0
    # n_jobs=1: avoids a real threading deadlock seen between xgboost and torch
    # in the same process.
    clf = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=4,
        eval_metric="logloss",
        n_jobs=1,
        scale_pos_weight=weight,
    )
    return Pipeline([("pre", pre), ("clf", clf)])

def evaluate(train_df, label):
    """Fits, prints the four metrics, and returns them -- returning them (not just
    printing) is what lets the next cell react to the actual numbers instead of
    assuming one outcome."""
    pipe = make_pipeline(train_df[TARGET])
    pipe.fit(train_df[CAT_COLS + NUM_COLS], train_df[TARGET])
    preds = pipe.predict(real_test[CAT_COLS + NUM_COLS])
    metrics = {
        "accuracy": accuracy_score(real_test[TARGET], preds),
        "precision": precision_score(real_test[TARGET], preds, zero_division=0),
        "recall": recall_score(real_test[TARGET], preds, zero_division=0),
        "f1": f1_score(real_test[TARGET], preds, zero_division=0),
    }
    print(
        f"{label:22s} accuracy={metrics['accuracy']:.3f}  "
        f"precision={metrics['precision']:.3f}  "
        f"recall={metrics['recall']:.3f}  "
        f"f1={metrics['f1']:.3f}"
    )
    return metrics

real_metrics = evaluate(real_train, "trained on REAL")
synthetic_metrics = evaluate(synthetic_table, "trained on SYNTHETIC")

In [ ]:
# A plain-language read of whatever REAL and SYNTHETIC actually scored above --
# generated from real_metrics/synthetic_metrics, not written for one assumed
# outcome, since which of these cases you land in changes what the result is
# actually telling you.
if real_metrics["f1"] == 0:
    print("Even the REAL-trained model found essentially nothing here (f1 = 0.000, recall = 0.000).")
    print("That's a property of this task at this sample size and tuning budget, not something")
    print("synthetic data could have fixed -- with no real signal to match in the first place, this")
    print("run's real-vs-synthetic comparison can't say much either way. Worth revisiting with")
    print("more tuning trials once the REAL-trained model itself finds something.")
elif synthetic_metrics["recall"] == 0:
    print(f"REAL found some signal (recall={real_metrics['recall']:.3f}, f1={real_metrics['f1']:.3f}) --")
    print("weak, but real. SYNTHETIC found none (recall = 0.000): a model trained on it is behaving")
    print("like it predicts \"no\" for every customer, not a subtle underperformance.")
    print("On this run, this synthetic table did not transfer for the actual task it was built for.")
elif synthetic_metrics["f1"] >= real_metrics["f1"] * 0.8:
    print(f"SYNTHETIC (f1={synthetic_metrics['f1']:.3f}) came close to, or matched, REAL")
    print(f"(f1={real_metrics['f1']:.3f}) on this run -- a genuinely encouraging transfer result")
    print("for this task, at this tuning budget.")
else:
    gap = real_metrics["f1"] - synthetic_metrics["f1"]
    print(f"SYNTHETIC (f1={synthetic_metrics['f1']:.3f}) underperformed REAL (f1={real_metrics['f1']:.3f})")
    print(f"by {gap:.3f} f1 -- partial transfer: the synthetic-trained model found some signal,")
    print("just less of it than training on real data directly would have.")

Both numbers are tested on the same real, held-out customers -- only the training data changes. Notice the accuracy figures first, and then notice how little they say: with a positive rate this low, a model that predicts "no" for every customer already scores well above 90% accuracy while catching none of the customers who actually come back. Precision, recall, and F1 (read above, against whichever of the cases above this run actually landed in) are what actually distinguish a model that learned something from one that didn't, which is the same rare-class lesson Week 3 taught, showing up here in a place accuracy alone would have hidden it completely.

What this comparison licenses is narrow, and it's worth being precise about the boundary. A close result here says this model, trained this way, on this target column, transfers from synthetic to real. It says nothing about a different target, a different model family, or an exploratory analysis somebody runs later on the same synthetic file -- exactly the limit this week's first lesson named, because CTGAN and TVAE are general-purpose models with no validity guarantee outside what was actually tested.

---

## What this does, and does not, protect

The synthetic table above is safe to hand to someone who can't see the real one, for the specific purpose it was measured against. It is not anonymous in some absolute sense, and its privacy protection is a property of this dataset and this configuration, not of the method in general.

What it protects: individual customers. A well-tuned model makes it hard to point at a synthetic row and recover a specific real person, and the k-anonymity check above is one way of measuring how hard.

What it does not protect: the organisation's own aggregate patterns. Conversion rate, average basket value, the shape of the payment-type mix, the seasonality in order volume -- all of that survives into the synthetic table by design, because matching those patterns is the entire point of the exercise. A synthetic dataset protects the people in the data more than it protects the business the data describes.

**Also in Week 5**: when to use synthetic data at all, how a synthetic dataset actually gets made, and what the platform's metrics are measuring and why they trade off against each other -- this notebook assumes all three.